In [1]:
import pandas as pd
import os
import numpy as np
import librosa
import time
from sklearn.model_selection import train_test_split

In [2]:
# 데이터프레임 불러오기
fifth_df = pd.read_csv(r"C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 음성 데이터셋\5차년도.csv", encoding='cp949')
fifth_df

,wav_id,발화문,상황,1번 감정,1번 감정세기,2번 감정,2번 감정세기,3번 감정,3번 감정세기,4번 감정,4번감정세기,5번 감정,5번 감정세기,나이,성별
0,5ed10dbc2880d70f286121c3,개를 예쁘다고 사놓고 끝까지 키우지도 않고 버리는 사람들이 엄청 많아졌대.,disgust,Angry,2,Angry,2,Angry,2,Angry,2,Angry,2,33,female
1,5ecb60ef9aa8ea0eec53edb1,지금도 그대로 있어. 치우는 사람이 없어.,disgust,Neutral,0,Disgust,2,Sadness,2,Disgust,2,Disgust,1,48,female
2,5f052858b140144dfcfef768,맞아. 무기력증인 것 같아. 한동안 정말 바빴었거든.,sad,Sadness,2,Sadness,1,Sadness,2,Sadness,2,Sadness,2,48,female
3,5f0e72c4b140144dfcff3fa5,오늘이 발표날인데 연락이 없더라고. 그래서 알아봤더니 명단에 내 이름이 없대.,sad,Sadness,1,Sadness,2,Sadness,2,Sadness,2,Sadness,2,48,female
4,5ed10ddd7e21a10eee2537ce,그치. 개 키우는 사람이 늘어나니까 그만큼 버리는 사람도 늘어나는 거야!,disgust,Angry,2,Angry,2,Angry,2,Angry,2,Sadness,2,33,female
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10006,5f05ffd5b140144dfcff01c7,너의 말 대로 약속 장소를 옮겨 보는 것도 좋은 방법 같아. 나는 취소할 생각만 했...,fear,Neutral,0,Happiness,1,Sadness,1,Neutral,0,Neutral,0,46,female
10007,5ec53bf82880d70f28611de0,산책하는 게 나을 것 같아. 이제부터 산책 좀 해볼게.,sad,Neutral,0,Happiness,1,Happiness,1,Neutral,0,Sadness,1,48,female
10008,5f0b2b5ab140144dfcff2759,너의 말을 듣고보니까 약속장소를 옮겨보는 것도 좋은 방법 같아.,fear,Neutral,0,Happiness,1,Sadness,1,Neutral,0,Neutral,0,46,female
10009,5f0b2b75b140144dfcff275a,오늘 친구한테 전화해서 안좋은 상황이니까 약속을 미룰지 약속 장소를 변경할지 상의해...,fear,Neutral,0,Fear,1,Sadness,1,Neutral,0,Fear,1,46,female


In [3]:
print(fifth_df.columns)

Index(['wav_id', '발화문', '상황', '1번 감정', '1번 감정세기', '2번 감정', '2번 감정세기', '3번 감정',
       '3번 감정세기', '4번 감정', '4번감정세기', '5번 감정', '5번 감정세기', '나이', '성별'],
      dtype='object')


In [4]:
# 필요없는 칼럼 제거
drop_columns = ['1번 감정','1번 감정세기','2번 감정','2번 감정세기','3번 감정','3번 감정세기','4번 감정','4번감정세기','5번 감정','5번 감정세기']
fifth_df.drop(columns = drop_columns, axis=1, inplace=True)
fifth_df

,wav_id,발화문,상황,나이,성별
0,5ed10dbc2880d70f286121c3,개를 예쁘다고 사놓고 끝까지 키우지도 않고 버리는 사람들이 엄청 많아졌대.,disgust,33,female
1,5ecb60ef9aa8ea0eec53edb1,지금도 그대로 있어. 치우는 사람이 없어.,disgust,48,female
2,5f052858b140144dfcfef768,맞아. 무기력증인 것 같아. 한동안 정말 바빴었거든.,sad,48,female
3,5f0e72c4b140144dfcff3fa5,오늘이 발표날인데 연락이 없더라고. 그래서 알아봤더니 명단에 내 이름이 없대.,sad,48,female
4,5ed10ddd7e21a10eee2537ce,그치. 개 키우는 사람이 늘어나니까 그만큼 버리는 사람도 늘어나는 거야!,disgust,33,female
...,...,...,...,...,...
10006,5f05ffd5b140144dfcff01c7,너의 말 대로 약속 장소를 옮겨 보는 것도 좋은 방법 같아. 나는 취소할 생각만 했...,fear,46,female
10007,5ec53bf82880d70f28611de0,산책하는 게 나을 것 같아. 이제부터 산책 좀 해볼게.,sad,48,female
10008,5f0b2b5ab140144dfcff2759,너의 말을 듣고보니까 약속장소를 옮겨보는 것도 좋은 방법 같아.,fear,46,female
10009,5f0b2b75b140144dfcff275a,오늘 친구한테 전화해서 안좋은 상황이니까 약속을 미룰지 약속 장소를 변경할지 상의해...,fear,46,female


In [21]:
# 데이터프레임에 음성파일 경로 추가
AUDIO_PATH = r"C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 음성 데이터셋\5차_wav"
fifth_df['file_path'] = fifth_df['wav_id'].apply(lambda x: os.path.join(AUDIO_PATH, x + '.wav'))
fifth_df

,wav_id,발화문,상황,나이,성별,file_path,file_exists
0,5ed10dbc2880d70f286121c3,개를 예쁘다고 사놓고 끝까지 키우지도 않고 버리는 사람들이 엄청 많아졌대.,disgust,33,female,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...,False
1,5ecb60ef9aa8ea0eec53edb1,지금도 그대로 있어. 치우는 사람이 없어.,disgust,48,female,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...,False
2,5f052858b140144dfcfef768,맞아. 무기력증인 것 같아. 한동안 정말 바빴었거든.,sad,48,female,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...,False
3,5f0e72c4b140144dfcff3fa5,오늘이 발표날인데 연락이 없더라고. 그래서 알아봤더니 명단에 내 이름이 없대.,sad,48,female,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...,False
4,5ed10ddd7e21a10eee2537ce,그치. 개 키우는 사람이 늘어나니까 그만큼 버리는 사람도 늘어나는 거야!,disgust,33,female,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...,False
...,...,...,...,...,...,...,...
10006,5f05ffd5b140144dfcff01c7,너의 말 대로 약속 장소를 옮겨 보는 것도 좋은 방법 같아. 나는 취소할 생각만 했...,fear,46,female,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...,False
10007,5ec53bf82880d70f28611de0,산책하는 게 나을 것 같아. 이제부터 산책 좀 해볼게.,sad,48,female,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...,False
10008,5f0b2b5ab140144dfcff2759,너의 말을 듣고보니까 약속장소를 옮겨보는 것도 좋은 방법 같아.,fear,46,female,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...,False
10009,5f0b2b75b140144dfcff275a,오늘 친구한테 전화해서 안좋은 상황이니까 약속을 미룰지 약속 장소를 변경할지 상의해...,fear,46,female,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...,False


In [25]:
# csv와 음성파일 매칭시켜 음성 파일이 존재하지 않는 csv 제거
# 파일 존재여부 확인
def check_file_exists(file_path):
    if pd.isna(file_path): 
        return False
    return os.path.exists(file_path)

# file_exists column을 새로 만들어 file_path가 True인지 False인지 데이터프레임에 추가
fifth_df['file_exists'] = fifth_df['file_path'].apply(check_file_exists)

# file_exists가 True인 row만 선택해서 df_clean에 copy
df_clean = fifth_df[fifth_df['file_exists']].copy()
# file_exists column drop
df_clean.drop(columns=['file_exists'], inplace=True)

print(f"제거 전 갯수: {len(fifth_df)}")
print(f"제거 후 갯수: {len(df_clean)}")
print(f"제거된 갯수: {len(fifth_df) - len(df_clean)}")

제거 전 갯수: 10011
제거 후 갯수: 10011
제거된 갯수: 0


In [27]:
df_clean

,wav_id,발화문,상황,나이,성별,file_path
0,5ed10dbc2880d70f286121c3,개를 예쁘다고 사놓고 끝까지 키우지도 않고 버리는 사람들이 엄청 많아졌대.,disgust,33,female,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...
1,5ecb60ef9aa8ea0eec53edb1,지금도 그대로 있어. 치우는 사람이 없어.,disgust,48,female,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...
2,5f052858b140144dfcfef768,맞아. 무기력증인 것 같아. 한동안 정말 바빴었거든.,sad,48,female,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...
3,5f0e72c4b140144dfcff3fa5,오늘이 발표날인데 연락이 없더라고. 그래서 알아봤더니 명단에 내 이름이 없대.,sad,48,female,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...
4,5ed10ddd7e21a10eee2537ce,그치. 개 키우는 사람이 늘어나니까 그만큼 버리는 사람도 늘어나는 거야!,disgust,33,female,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...
...,...,...,...,...,...,...
10006,5f05ffd5b140144dfcff01c7,너의 말 대로 약속 장소를 옮겨 보는 것도 좋은 방법 같아. 나는 취소할 생각만 했...,fear,46,female,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...
10007,5ec53bf82880d70f28611de0,산책하는 게 나을 것 같아. 이제부터 산책 좀 해볼게.,sad,48,female,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...
10008,5f0b2b5ab140144dfcff2759,너의 말을 듣고보니까 약속장소를 옮겨보는 것도 좋은 방법 같아.,fear,46,female,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...
10009,5f0b2b75b140144dfcff275a,오늘 친구한테 전화해서 안좋은 상황이니까 약속을 미룰지 약속 장소를 변경할지 상의해...,fear,46,female,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...


In [29]:
#상황 column을 감정으로 변경
df_clean.rename(columns={'상황':'감정'}, inplace=True)
#나이,성별 column drop => mfcc 중복 최소화
df_clean.drop(columns=['나이','성별'], inplace=True)
df_clean

,wav_id,발화문,감정,file_path
0,5ed10dbc2880d70f286121c3,개를 예쁘다고 사놓고 끝까지 키우지도 않고 버리는 사람들이 엄청 많아졌대.,disgust,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...
1,5ecb60ef9aa8ea0eec53edb1,지금도 그대로 있어. 치우는 사람이 없어.,disgust,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...
2,5f052858b140144dfcfef768,맞아. 무기력증인 것 같아. 한동안 정말 바빴었거든.,sad,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...
3,5f0e72c4b140144dfcff3fa5,오늘이 발표날인데 연락이 없더라고. 그래서 알아봤더니 명단에 내 이름이 없대.,sad,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...
4,5ed10ddd7e21a10eee2537ce,그치. 개 키우는 사람이 늘어나니까 그만큼 버리는 사람도 늘어나는 거야!,disgust,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...
...,...,...,...,...
10006,5f05ffd5b140144dfcff01c7,너의 말 대로 약속 장소를 옮겨 보는 것도 좋은 방법 같아. 나는 취소할 생각만 했...,fear,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...
10007,5ec53bf82880d70f28611de0,산책하는 게 나을 것 같아. 이제부터 산책 좀 해볼게.,sad,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...
10008,5f0b2b5ab140144dfcff2759,너의 말을 듣고보니까 약속장소를 옮겨보는 것도 좋은 방법 같아.,fear,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...
10009,5f0b2b75b140144dfcff275a,오늘 친구한테 전화해서 안좋은 상황이니까 약속을 미룰지 약속 장소를 변경할지 상의해...,fear,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...


In [31]:
# MFCC : 음성 데이터를 특징 벡터화해주는 알고리즘
# file_path를 받아 특징 추출 함수 정의
def extract_features(file_path):
    try:
        # y : 음성데이터의 numpy 배열
        # sr :샘플링 속도 / sr=16000 : 초당 16000개의 샘플을 가지고 있는 데이터 / 16000인 이유 : 사람의 목소리는 대부분 16000Hz 안에 포함
        # liborsa : 음성 파일 불러오기
        y, sr = librosa.load(file_path, sr=16000) 
    except Exception as e:
        print(f"Error loading {file_path}: {e}")
        return None
    # n_mfcc:mfcc의 개수를 정해주는 파라미터
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40) 
    mfccs_mean = np.mean(mfccs.T, axis=0) 
    
    return mfccs_mean

print("특징 추출")
start_time = time.time()

df_clean['features'] = df_clean['file_path'].apply(extract_features)

end_time = time.time()
print(f"--- 특징 추출 완료. 총 소요 시간 : {end_time - start_time:.2f}초 ---")

# 특징 추출 중 오류(None)가 발생한 행은 제거
df_clean.dropna(subset=['features'], inplace=True)

# 최종 확인
print("\n[추출 결과 확인]")
print(df_clean.head())
print(f"처리된 데이터 수: {len(df_clean)}개")
print(f"특징 벡터 크기: {df_clean['features'].iloc[0].shape if len(df_clean) > 0 else 'N/A'}")

특징 추출


C:\Users\user\anaconda3\Lib\site-packages\paramiko\pkey.py:82: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from this module in 48.0.0.
  "cipher": algorithms.TripleDES,
C:\Users\user\anaconda3\Lib\site-packages\paramiko\transport.py:219: CryptographyDeprecationWarning: Blowfish has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.Blowfish and will be removed from this module in 45.0.0.
  "class": algorithms.Blowfish,
C:\Users\user\anaconda3\Lib\site-packages\paramiko\transport.py:243: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from this module in 48.0.0.
  "class": algorithms.TripleDES,


--- 특징 추출 완료. 총 소요 시간 : 567.58초 ---

[추출 결과 확인]
                     wav_id                                          발화문  \
0  5ed10dbc2880d70f286121c3    개를 예쁘다고 사놓고 끝까지 키우지도 않고 버리는 사람들이 엄청 많아졌대.   
1  5ecb60ef9aa8ea0eec53edb1                      지금도 그대로 있어. 치우는 사람이 없어.   
2  5f052858b140144dfcfef768                맞아. 무기력증인 것 같아. 한동안 정말 바빴었거든.   
3  5f0e72c4b140144dfcff3fa5  오늘이 발표날인데 연락이 없더라고. 그래서 알아봤더니 명단에 내 이름이 없대.   
4  5ed10ddd7e21a10eee2537ce     그치. 개 키우는 사람이 늘어나니까 그만큼 버리는 사람도 늘어나는 거야!   

        감정                                          file_path  \
0  disgust  C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...   
1  disgust  C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...   
2      sad  C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...   
3      sad  C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...   
4  disgust  C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...   

                                            features  
0  [-388.3951, 74.730896, 13.282206, 7.2658453, -

In [32]:
df_clean

,wav_id,발화문,감정,file_path,features
0,5ed10dbc2880d70f286121c3,개를 예쁘다고 사놓고 끝까지 키우지도 않고 버리는 사람들이 엄청 많아졌대.,disgust,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...,"[-388.3951, 74.730896, 13.282206, 7.2658453, -..."
1,5ecb60ef9aa8ea0eec53edb1,지금도 그대로 있어. 치우는 사람이 없어.,disgust,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...,"[-366.50607, 79.11673, 3.719925, 25.20848, 14...."
2,5f052858b140144dfcfef768,맞아. 무기력증인 것 같아. 한동안 정말 바빴었거든.,sad,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...,"[-381.88306, 132.82869, -34.271587, 29.955187,..."
3,5f0e72c4b140144dfcff3fa5,오늘이 발표날인데 연락이 없더라고. 그래서 알아봤더니 명단에 내 이름이 없대.,sad,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...,"[-469.61, 133.62344, 31.879255, 26.25098, 4.16..."
4,5ed10ddd7e21a10eee2537ce,그치. 개 키우는 사람이 늘어나니까 그만큼 버리는 사람도 늘어나는 거야!,disgust,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...,"[-385.09213, 65.325356, 10.039982, 12.324338, ..."
...,...,...,...,...,...
10006,5f05ffd5b140144dfcff01c7,너의 말 대로 약속 장소를 옮겨 보는 것도 좋은 방법 같아. 나는 취소할 생각만 했...,fear,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...,"[-319.5962, 68.805725, -3.5958617, -4.518435, ..."
10007,5ec53bf82880d70f28611de0,산책하는 게 나을 것 같아. 이제부터 산책 좀 해볼게.,sad,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...,"[-409.45132, 72.56055, 0.49839407, 19.704346, ..."
10008,5f0b2b5ab140144dfcff2759,너의 말을 듣고보니까 약속장소를 옮겨보는 것도 좋은 방법 같아.,fear,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...,"[-332.28665, 62.436436, -6.3532276, 6.9299006,..."
10009,5f0b2b75b140144dfcff275a,오늘 친구한테 전화해서 안좋은 상황이니까 약속을 미룰지 약속 장소를 변경할지 상의해...,fear,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...,"[-306.4688, 48.18986, -6.252661, 20.736578, -6..."


In [40]:
#추출된 40개의 음성 특징을 column으로 분리
features_df = pd.DataFrame(df_clean['features'].tolist(), index=df_clean.index)
# 기존의 데이터프레임과 합치기
df_final_csv = pd.concat([df_clean.drop('features', axis=1), features_df], axis=1)
# column이름 정리
new_columns = {i: f'feature_{i}' for i in range(features_df.shape[1])}
df_final_csv.rename(columns=new_columns, inplace=True)
#csv파일로 새로 저장하기
output_file_path = "preprocessing_fifth_df.csv"
df_final_csv.to_csv(output_file_path, index=False) 

In [38]:
df_final_csv

,wav_id,발화문,감정,file_path,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,...,feature_30,feature_31,feature_32,feature_33,feature_34,feature_35,feature_36,feature_37,feature_38,feature_39
0,5ed10dbc2880d70f286121c3,개를 예쁘다고 사놓고 끝까지 키우지도 않고 버리는 사람들이 엄청 많아졌대.,disgust,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...,-388.395111,74.730896,13.282206,7.265845,-3.945615,-3.621769,...,1.045132,4.241167,-3.240234,-1.338474,-5.396071,-3.857998,-7.007072,-0.223521,-4.782117,-1.663520
1,5ecb60ef9aa8ea0eec53edb1,지금도 그대로 있어. 치우는 사람이 없어.,disgust,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...,-366.506073,79.116730,3.719925,25.208481,14.733514,8.084488,...,-0.309401,0.062468,-1.239123,-0.164439,-3.376769,-0.946695,-2.648921,-2.935347,-5.095393,-2.514532
2,5f052858b140144dfcfef768,맞아. 무기력증인 것 같아. 한동안 정말 바빴었거든.,sad,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...,-381.883057,132.828690,-34.271587,29.955187,15.027156,-10.020905,...,-2.886052,-3.397950,-4.847836,-3.933729,-4.368754,-1.693319,-4.063304,-5.023028,-5.322674,-1.867804
3,5f0e72c4b140144dfcff3fa5,오늘이 발표날인데 연락이 없더라고. 그래서 알아봤더니 명단에 내 이름이 없대.,sad,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...,-469.609985,133.623444,31.879255,26.250980,4.164637,-26.347017,...,-7.043136,-4.875816,-5.937447,-4.716726,-4.043817,-4.590986,-6.348584,-4.372539,-5.783140,-3.003197
4,5ed10ddd7e21a10eee2537ce,그치. 개 키우는 사람이 늘어나니까 그만큼 버리는 사람도 늘어나는 거야!,disgust,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...,-385.092133,65.325356,10.039982,12.324338,6.501280,-0.105276,...,5.963353,3.201367,-0.111456,0.290162,-2.618307,-3.568064,-2.684454,-1.327447,-5.922699,0.011921
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10006,5f05ffd5b140144dfcff01c7,너의 말 대로 약속 장소를 옮겨 보는 것도 좋은 방법 같아. 나는 취소할 생각만 했...,fear,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...,-319.596191,68.805725,-3.595862,-4.518435,1.936625,-22.899960,...,2.303281,6.633930,3.129032,5.774830,2.185595,2.835067,0.625125,0.729478,-0.716560,-1.097932
10007,5ec53bf82880d70f28611de0,산책하는 게 나을 것 같아. 이제부터 산책 좀 해볼게.,sad,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...,-409.451324,72.560547,0.498394,19.704346,18.097246,-12.828385,...,-7.386422,-1.302072,-2.113776,-3.100656,-2.735099,-3.545804,-4.676533,0.881080,-5.836530,-1.227656
10008,5f0b2b5ab140144dfcff2759,너의 말을 듣고보니까 약속장소를 옮겨보는 것도 좋은 방법 같아.,fear,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...,-332.286652,62.436436,-6.353228,6.929901,-11.079716,-6.073962,...,8.785499,0.397186,0.771372,-4.725410,-2.325348,3.184555,-1.070802,-4.237032,-3.917394,-2.565317
10009,5f0b2b75b140144dfcff275a,오늘 친구한테 전화해서 안좋은 상황이니까 약속을 미룰지 약속 장소를 변경할지 상의해...,fear,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...,-306.468811,48.189861,-6.252661,20.736578,-6.269107,-9.850458,...,7.957385,0.345753,-0.079380,-5.665269,-3.094472,5.181507,-3.119624,-5.061707,-3.580066,-1.362962
